In [0]:
from pyspark.sql.functions import col


In [0]:
def load_bronze_table(table_name):

    df = spark.table(f"e_comm_databricks.bronze.olist_{table_name}")

    return df

In [0]:
def deduplicate(df, columns):

    before_count = df.count()

    df_clean = df.dropDuplicates(columns)

    after_count = df_clean.count()

    removed = before_count - after_count

    return df_clean, removed


In [0]:
def remove_nulls(df, columns):

    before_count = df.count()

    for c in columns:
        df = df.filter(col(c).isNotNull())

    after_count = df.count()

    removed = before_count - after_count

    return df, removed


In [0]:
def write_silver(df, table_name):

    df.write \
      .format("delta") \
      .mode("overwrite") \
      .saveAsTable(f"e_comm_databricks.silver.{table_name}")


In [0]:
silver_tables_config = {

   "orders": {
      "dedup_columns": ["order_id"],
      "not_null": ["order_id","customer_id"]
   },

   "order_items": {
      "dedup_columns": ["order_id","order_item_id"],
      "not_null": ["order_id","product_id"]
   },

   "order_payments": {
      "dedup_columns": ["order_id","payment_sequential"],
      "not_null": ["order_id"]
   },

   "order_reviews": {
      "dedup_columns": ["review_id"],
      "not_null": ["review_id"]
   }

}


In [0]:
for table_name, rules in silver_tables_config.items():

    print(f"Processing {table_name}")

    df = load_bronze_table(table_name)

    df = deduplicate(df, rules["dedup_columns"])

    df = remove_nulls(df, rules["not_null"])

    write_silver(df, table_name)


In [0]:
df = load_bronze_table("orders")

df = deduplicate(df, ["order_id"])

df = remove_nulls(df, ["order_id","customer_id"])

write_silver(df, "orders")


In [0]:
dq_audit_log = []


In [0]:
for table_name, rules in silver_tables_config.items():

    print(f"Processing {table_name}")

    df = load_bronze_table(table_name)

    initial_count = df.count()

    df, dup_removed = deduplicate(df, rules["dedup_columns"])

    df, null_removed = remove_nulls(df, rules["not_null"])

    final_count = df.count()

    write_silver(df, table_name)

    dq_audit_log.append(
        (table_name, initial_count, dup_removed, null_removed, final_count)
    )


In [0]:
dq_report = spark.createDataFrame(
    dq_audit_log,
    ["table","rows_before","duplicates_removed","null_rows_removed","rows_after"]
)

display(dq_report)


"add automatic relationship validation"

"add automatic relationship validation"

order_items → orders relationship

payments → orders relationship

reviews → orders relationship

In [0]:
relationship_config = [

   {
      "child_table": "order_items",
      "parent_table": "orders",
      "join_column": "order_id"
   },

   {
      "child_table": "order_payments",
      "parent_table": "orders",
      "join_column": "order_id"
   },

   {
      "child_table": "order_reviews",
      "parent_table": "orders",
      "join_column": "order_id"
   }

]


In [0]:
def validate_relationship(child_table, parent_table, column):

    df_child = spark.table(f"e_comm_databricks.silver.{child_table}")
    df_parent = spark.table(f"e_comm_databricks.silver.{parent_table}")

    invalid_rows = df_child.join(
        df_parent.select(column),
        on=column,
        how="left_anti"
    )

    invalid_count = invalid_rows.count()

    return invalid_count


In [0]:
relationship_results = []

for rel in relationship_config:

    count = validate_relationship(
        rel["child_table"],
        rel["parent_table"],
        rel["join_column"]
    )

    relationship_results.append(
        (rel["child_table"], rel["parent_table"], rel["join_column"], count)
    )


In [0]:
relationship_report = spark.createDataFrame(
    relationship_results,
    ["child_table","parent_table","column","invalid_rows"]
)

display(relationship_report)
